<a href="https://colab.research.google.com/github/ngoan22mse23088/ArtificialIntelligence/blob/master/Lung-Cance-Detection-Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**About The Project**


This project focuses on lung cancer detection using CT scan images from the IQ-OTH/NCCD Lung Cancer Dataset, collected by The Iraq Oncology Teaching Hospital / National Center for Cancer Diseases. The dataset includes three classes: Normal, Benign, and Malignant.

A ResNet-18 model pretrained on ImageNet was used through transfer learning. The final fully connected layer was replaced to match the three output classes, and all layers were fine-tuned to adapt the model to medical imaging data. The model was trained using CrossEntropyLoss, the AdamW optimizer, and a ReduceLROnPlateau learning rate scheduler.

Performance was evaluated using test loss, accuracy, classification report, and a confusion matrix. The fine-tuned ResNet-18 achieved strong classification performance, demonstrating the effectiveness of transfer learning for automated lung cancer detection.

**Re-Check Images**

In [ ]:
import kagglehub
from pathlib import Path
from collections import defaultdict
import json
from PIL import Image
import numpy as np

# Download dataset
path = kagglehub.dataset_download("adityamahimkar/iqothnccd-lung-cancer-dataset")
print("Path to dataset files:", path)

# ── Tự động tìm ảnh trong path vừa download ──
root = Path(path)
ALLOWED_EXTS   = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}
EXPECTED_MODES = {"RGB", "L"}

image_files = [f for f in root.rglob("*")
               if f.is_file() and f.suffix.lower() in ALLOWED_EXTS]

print(f"\n📁 Dataset: {root}")
print(f"🖼️  Tổng số ảnh: {len(image_files)}")

if len(image_files) == 0:
    all_files = list(root.rglob("*.*"))
    exts = set(f.suffix.lower() for f in all_files if f.is_file())
    print(f"⚠️  Không tìm thấy ảnh! Extensions thực tế: {exts}")
    for f in all_files[:10]:
        print(f"  {f.relative_to(root)}")
else:
    errors        = []
    size_counter  = defaultdict(int)
    mode_counter  = defaultdict(int)
    class_counter = defaultdict(int)
    ext_counter   = defaultdict(int)

    for i, p in enumerate(image_files, 1):
        if i % 300 == 0:
            print(f"  ... {i}/{len(image_files)}")
        issues = []
        class_counter[p.parent.name] += 1
        ext_counter[p.suffix.lower()] += 1

        if p.stat().st_size < 1024:
            issues.append(f"File quá nhỏ ({p.stat().st_size} bytes)")

        try:
            with Image.open(p) as img:
                w, h, mode = img.width, img.height, img.mode
                size_counter[f"{w}x{h}"] += 1
                mode_counter[mode] += 1
                if mode not in EXPECTED_MODES:
                    issues.append(f"Mode không hợp lệ: {mode}")
                if np.array(img).std() < 1.0:
                    issues.append("Ảnh một màu (std≈0)")
        except Exception as e:
            issues.append(f"Không đọc được: {e}")

        if issues:
            errors.append({"file": str(p.relative_to(root)), "issues": issues})

    total = len(image_files)
    n_ok  = total - len(errors)

    print(f"\n{'═'*55}")
    print(f"  Tổng ảnh    : {total}")
    print(f"  ✅ Đạt chuẩn : {n_ok}  ({n_ok/total*100:.1f}%)")
    print(f"  ❌ Có vấn đề : {len(errors)}  ({len(errors)/total*100:.1f}%)")

    print(f"\n  📂 PHÂN BỐ THEO CLASS:")
    max_c = max(class_counter.values())
    for cls, cnt in sorted(class_counter.items()):
        bar = "█" * int(cnt / max_c * 25)
        print(f"    {cls:<30} {cnt:>4}  {bar}")

    print(f"\n  📐 KÍCH THƯỚC PHỔ BIẾN:")
    for size, cnt in sorted(size_counter.items(), key=lambda x: -x[1])[:8]:
        print(f"    {size:<15} {cnt:>4} ảnh")

    print(f"\n  🎨 MODE MÀU:")
    for mode, cnt in sorted(mode_counter.items(), key=lambda x: -x[1]):
        mark = "✅" if mode in EXPECTED_MODES else "❌"
        print(f"    {mode:<10} {cnt:>4} ảnh  {mark}")

    print(f"\n  📄 ĐỊNH DẠNG FILE:")
    for ext, cnt in sorted(ext_counter.items(), key=lambda x: -x[1]):
        print(f"    {ext:<10} {cnt:>4} ảnh")

    if errors:
        print(f"\n  ❌ MẪU LỖI (tối đa 20):")
        for e in errors[:20]:
            print(f"  [{e['file']}]")
            for iss in e["issues"]:
                print(f"    → {iss}")
        out = Path("/kaggle/working/error_images.json")
        out.write_text(json.dumps(errors, ensure_ascii=False, indent=2))
        print(f"\n  📋 Đã lưu lỗi → {out}")
    else:
        print(f"\n  🎉 Tất cả ảnh đều đạt chuẩn!")
    print(f"{'═'*55}")

Using Colab cache for faster access to the 'iqothnccd-lung-cancer-dataset' dataset.
Path to dataset files: /kaggle/input/iqothnccd-lung-cancer-dataset
